### Library imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os
import numpy as np

### Training Model

In [2]:

print("Loading dataset...")
df = pd.read_csv("../data/SMSSpamCollection.csv", encoding="latin-1")

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    test_size=0.2, 
    random_state=42  # This makes it reproducible
)

print("Vectorizing text (TF-IDF)...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Training Naive Bayes classifier...")
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

print("Evaluating model...")
THRESHOLD = 0.20
probabilities = model.predict_proba(X_test_vectorized)[:, 1]
predictions = np.where(probabilities >= THRESHOLD, 'spam', 'ham')
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, pos_label='spam')

print("-" * 30)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print("-" * 30)

print("Exporting Model...")
# Saves the trained vectorizer and model to disk
os.makedirs("models", exist_ok=True)
joblib.dump(vectorizer, "../artifacts/models/tfidf_vectorizer.joblib")
joblib.dump(model, "../artifacts/models/naive_bayes_model.joblib")
print("Models saved to the artifacts directory.")




Loading dataset...
Splitting data...
Vectorizing text (TF-IDF)...
Training Naive Bayes classifier...
Evaluating model...
------------------------------
Accuracy: 98.39%
F1 Score: 93.92%
------------------------------
Exporting Model...
Models saved to the artifacts directory.


### Reports

In [3]:
print("Classification Report:")
print(classification_report(y_test, predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       965
        spam       0.95      0.93      0.94       150

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
[[958   7]
 [ 11 139]]
